# Stage 1 model

In [1]:
import pandas as pd
import plotly.express as px

from ambdes import Model, Runner, SimConfig, ambsys, ArrivalConfig

## Run model for 100 minutes with log messages

TODO: Convert this into logging tests which compare patient results with record in the vidigi logger.

In [2]:
ambsys_data = ambsys(
    csv_path="data/AmbSYS-to-Mar-2026-UI7FG.csv",
    org_code="RYF",
    month="3",
    year=2026,
)
ambsys_data

{'mean_iat_min': {'C1': 5.02646098412341,
  'C2': 1.042868823735545,
  'C3': 2.4551754482455177,
  'C4': 135.6838905775076},
 'mean_handover_time_min': 29.916666666666668,
 'p90_handover_time_min': 51.28333333333333,
 'sd_handover_time_min': 11.385314934097805}

In [3]:
arrival_config = ArrivalConfig(arrival_df="data/arrivals.csv")
display(arrival_config.arrival_df)
display(arrival_config.nspp_df)

,C1,C2,C3,C4
monday,25,310,180,40
tuesday,24,295,170,38
wednesday,24,295,170,38
thursday,24,295,170,38
friday,25,305,178,40
saturday,28,340,200,45
sunday,27,330,195,43


,t,mean_iat
0,0,2.594595
1,1440,2.732448
2,2880,2.732448
3,4320,2.732448
4,5760,2.627737
5,7200,2.349103
6,8640,2.420168


In [4]:
config = SimConfig(
    ambsys_data=ambsys_data,
    arrival_config=arrival_config
)
config.n_ambulances = 1
model = Model(run_number=0, config=config)
model.run()

In [5]:
log = model.logger.to_dataframe()

In [6]:
print(model.patients[0].__dict__)
log[log["entity_id"] == 1]

{'patient_id': 1, 'category': 'C2', 'call_timestamp': 101.54417229458636, 'response_time': None}


,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,3.034460,0,NaN
1,1,queue,ambulance_wait_begins,3.034460,0,NaN
2,1,resource_use,ambulance_assigned,3.034460,0,1.0
71,1,arrival_departure,arrival,101.544172,0,NaN
72,1,queue,ambulance_wait_begins,101.544172,0,NaN
87,1,resource_use_end,ambulance_available,126.294696,0,1.0
88,1,arrival_departure,depart,126.294696,0,NaN


In [7]:
print(model.patients[1].__dict__)
log[log["entity_id"] == 2]

{'patient_id': 2, 'category': 'C2', 'call_timestamp': 101.74401794352872, 'response_time': None}


,entity_id,event_type,event,time,run_number,resource_id
3,2,arrival_departure,arrival,13.420421,0,NaN
4,2,queue,ambulance_wait_begins,13.420421,0,NaN
73,2,arrival_departure,arrival,101.744018,0,NaN
74,2,queue,ambulance_wait_begins,101.744018,0,NaN
89,2,resource_use,ambulance_assigned,126.294696,0,1.0


## Run model for longer and inspect patient times

In [8]:
config = SimConfig(
    ambsys_data=ambsys_data,
    arrival_config=arrival_config,
    warm_up_period=0,
    data_collection_period=10080,  # One week
)
model = Model(run_number=0, config=config)
model.run()

In [9]:
df = pd.DataFrame(
    {
        "response_time": [p.response_time for p in model.patients],
        "category": [p.category for p in model.patients],
    }
)

fig = px.histogram(
    df,
    x="response_time",
    facet_col="category",
    nbins=50,
    category_orders={"category": ["C1", "C2", "C3", "C4"]},
    labels={
        "response_time": "Response time (minutes)",
        "category": "Category",
    },
    title="Distribution of response times by category",
)

# fig.update_yaxes(
#    matches=None,
#    showticklabels=True,
# )
fig.layout.yaxis.title.text = "Number of patients"

fig.show()

In [10]:
for cat in ["C1", "C2", "C3", "C4"]:
    fig = px.histogram(
        df[df["category"] == cat],
        x="response_time",
        nbins=20,
        title=f"Response times: {cat}",
        labels={"response_time": "Response time (minutes)"},
    )
    fig.update_yaxes(title_text="Number of patients")
    fig.show()

## Average results

In [11]:
runner = Runner(config)

In [12]:
results = runner.run_reps()

/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)
/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)
/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecate

In [13]:
results["patients"]

,run,patient_id,category,call_timestamp,response_time
0,0,1,C3,2.700278,11.237279
1,0,2,C2,5.729027,19.055716
2,0,3,C3,7.181276,0.835666
3,0,4,C2,11.081455,9.576282
4,0,5,C3,11.378460,1.972919
...,...,...,...,...,...
19407,4,3974,C3,10073.930060,NaN
19408,4,3975,C3,10076.477597,NaN
19409,4,3976,C2,10077.855483,1.231421
19410,4,3977,C2,10079.108476,NaN


In [14]:
results["run"]

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,176,10.234737,0,NaN
1,C2,2118,9.892770,0,NaN
2,C3,1208,9.411465,0,NaN
3,C4,286,10.291139,0,NaN
4,all,<NA>,NaN,0,0.129895
5,C1,166,10.664535,1,NaN
6,C2,2061,9.867644,1,NaN
7,C3,1390,10.396935,1,NaN
8,C4,282,11.159477,1,NaN
9,all,<NA>,NaN,1,0.134113


In [15]:
results["overall"]

,category,mean_n_patients,mean_response_time,mean_utilisation
0,C1,173.2,10.269885,NaN
1,C2,2156.6,9.877646,NaN
2,C3,1270.8,10.013623,NaN
3,C4,281.8,10.097816,NaN
4,all,NaN,NaN,0.133196


In [16]:
config.n_ambulances = 1
config.data_collection_period = 50_000
config.log_to_console = False
runner = Runner(config)
results = runner.run_single(run_number=0)

/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)


In [17]:
results["run"]

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,927,19966.579609,0,NaN
1,C2,10873,24871.015999,0,NaN
2,C3,6168,23241.884425,0,NaN
3,C4,1394,25233.632020,0,NaN
4,all,<NA>,NaN,0,0.99988


In [18]:
results["patients"].head(20)

,run,patient_id,category,call_timestamp,response_time
0,0,1,C3,6.019644,11.237279
1,0,2,C2,10.672864,137.662732
2,0,3,C3,14.313891,215.454771
3,0,4,C2,15.946767,344.575984
4,0,5,C3,16.242647,444.709639
5,0,6,C3,17.539190,523.131718
6,0,7,C2,17.954737,647.930960
7,0,8,C3,21.881839,766.937978
8,0,9,C2,22.209387,891.360543
9,0,10,C4,27.045708,964.901587


In [19]:
results["model"].logger.to_dataframe().head(30)

,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,6.019644,0,NaN
1,1,queue,ambulance_wait_begins,6.019644,0,NaN
2,1,resource_use,ambulance_assigned,6.019644,0,1.0
3,2,arrival_departure,arrival,10.672864,0,NaN
4,2,queue,ambulance_wait_begins,10.672864,0,NaN
5,3,arrival_departure,arrival,14.313891,0,NaN
6,3,queue,ambulance_wait_begins,14.313891,0,NaN
7,4,arrival_departure,arrival,15.946767,0,NaN
8,4,queue,ambulance_wait_begins,15.946767,0,NaN
9,5,arrival_departure,arrival,16.242647,0,NaN
